In [14]:
# create loader
# load tests
# get tokenizer
# create processor
# preprocess one test example

In [15]:
import sys
sys.path.append("../")

In [16]:
# imports
from src.ner.loading import Loader
from pathlib import Path

In [17]:
# create loader
loader = Loader(path=Path("../data/entities.json"), test_k=0.2, seed=42)

In [18]:
# load tests
tests = loader.load_test_dataset()
tests[0]

{'text': 'Customer support escalated prefix Mr. with lastname Petrov because the email ivan.petrov@northwind-mail.net was tied to account number 11880073.',
 'entities': [{'end': 37, 'label': 'PREFIX', 'start': 34},
  {'end': 58, 'label': 'LASTNAME', 'start': 52},
  {'end': 107, 'label': 'EMAIL', 'start': 77},
  {'end': 143, 'label': 'ACCOUNT_NUMBER', 'start': 135}]}

In [19]:
# get tokenizer
from src.ner.models import Model

model = Model(name="microsoft/deberta-v3-base")
tokenizer = model.get_tokenizer()

The tokenizer you are loading from '/Users/ayeustsihneyeu/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [20]:
# create processor
from src.ner.processing import Processor

processor = Processor(
    tokenizer=tokenizer
)

example = tests[0]
example

{'text': 'Customer support escalated prefix Mr. with lastname Petrov because the email ivan.petrov@northwind-mail.net was tied to account number 11880073.',
 'entities': [{'end': 37, 'label': 'PREFIX', 'start': 34},
  {'end': 58, 'label': 'LASTNAME', 'start': 52},
  {'end': 107, 'label': 'EMAIL', 'start': 77},
  {'end': 143, 'label': 'ACCOUNT_NUMBER', 'start': 135}]}

In [21]:
text = example["text"]
text

'Customer support escalated prefix Mr. with lastname Petrov because the email ivan.petrov@northwind-mail.net was tied to account number 11880073.'

In [22]:
entities = sorted(example["entities"], key=lambda x: x["start"])
entities

[{'end': 37, 'label': 'PREFIX', 'start': 34},
 {'end': 58, 'label': 'LASTNAME', 'start': 52},
 {'end': 107, 'label': 'EMAIL', 'start': 77},
 {'end': 143, 'label': 'ACCOUNT_NUMBER', 'start': 135}]

In [23]:
tokens = tokenizer(
    text,
    truncation=True,
    max_length=256,
    return_offsets_mapping=True
)
tokens

{'input_ids': [1, 5030, 523, 30084, 25916, 945, 260, 275, 437, 8982, 69179, 401, 262, 871, 584, 10899, 260, 14851, 34994, 1683, 33546, 20039, 271, 3410, 260, 3163, 284, 4946, 264, 914, 496, 16863, 4835, 9743, 260, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'offset_mapping': [(0, 0), (0, 8), (8, 16), (16, 26), (26, 33), (33, 36), (36, 37), (37, 42), (42, 47), (47, 51), (51, 58), (58, 66), (66, 70), (70, 76), (76, 78), (78, 81), (81, 82), (82, 85), (85, 88), (88, 89), (89, 94), (94, 98), (98, 99), (99, 103), (103, 104), (104, 107), (107, 111), (111, 116), (116, 119), (119, 127), (127, 134), (134, 138), (138, 141), (141, 143), (143, 144), (0, 0)]}

In [24]:
from src.ner.processing import label2id

def preprocess(example: dict, tokenizer):

    text = example["text"]
    entities = sorted(example["entities"], key=lambda x: x["start"])

    encoding = tokenizer(
        text,
        truncation=True,
        max_length=256,
        return_offsets_mapping=True,
    )

    offsets = encoding["offset_mapping"]
    labels = []
    label_bios = []
    current_entity_idx = None

    for offset in offsets:
        start_char, end_char = offset

        # special tokens usually have offset (0, 0)
        if start_char == end_char == 0:
            labels.append(-100)
            current_entity_idx = None
            continue

        matched_entity_idx = None

        # search entity, where it is into offset
        for idx, ent in enumerate(entities):
            ent_start = ent["start"]
            ent_end = ent["end"]

            if start_char >= ent_start and end_char <= ent_end:
                matched_entity_idx = idx
                break

        if matched_entity_idx is None:
            labels.append(label2id["O"])
            label_bios.append("O")
            current_entity_idx = None
        else:
            ent = entities[matched_entity_idx]
            ent_label = ent["label"]

            # if it is first token of entity
            if current_entity_idx != matched_entity_idx:
                labels.append(label2id[f"B-{ent_label}"])
                label_bios.append(f"B-{ent_label}")
            else:
                labels.append(label2id[f"I-{ent_label}"])
                label_bios.append(f"I-{ent_label}")

            current_entity_idx = matched_entity_idx

    encoding["labels"] = labels
    encoding.pop("offset_mapping")
    return encoding, label_bios

In [25]:
res = preprocess(example=example, tokenizer=tokenizer)
print(res)

({'input_ids': [1, 5030, 523, 30084, 25916, 945, 260, 275, 437, 8982, 69179, 401, 262, 871, 584, 10899, 260, 14851, 34994, 1683, 33546, 20039, 271, 3410, 260, 3163, 284, 4946, 264, 914, 496, 16863, 4835, 9743, 260, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 0, 0, 0, 0, 0, 59, 0, 0, 0, 0, 0, 0, 0, 0, 33, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 0, 0, 0, 0, 0, 0, 3, 4, 0, -100]}, ['O', 'O', 'O', 'O', 'O', 'B-PREFIX', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-EMAIL', 'I-EMAIL', 'I-EMAIL', 'I-EMAIL', 'I-EMAIL', 'I-EMAIL', 'I-EMAIL', 'I-EMAIL', 'I-EMAIL', 'I-EMAIL', 'I-EMAIL', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ACCOUNT_NUMBER', 'I-ACCOUNT_NUMBER', 'O'])


In [27]:
# preprocess one test example
emb, lab = processor.preprocess(example=example, return_label_bios=True)
lab

['O',
 'O',
 'O',
 'O',
 'O',
 'B-PREFIX',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-EMAIL',
 'I-EMAIL',
 'I-EMAIL',
 'I-EMAIL',
 'I-EMAIL',
 'I-EMAIL',
 'I-EMAIL',
 'I-EMAIL',
 'I-EMAIL',
 'I-EMAIL',
 'I-EMAIL',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-ACCOUNT_NUMBER',
 'I-ACCOUNT_NUMBER',
 'O']